# 2s10s spread — proof of downstream UX

Goal of this notebook: demonstrate that with the warehouse populated, a future Claude Code session (or any analyst) can chart "X vs Y over time" with no fetching code.

We pull `FRED:DGS10` (10-year Treasury yield) and `FRED:DGS2` (2-year Treasury yield) and plot the 2s10s spread.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

import matplotlib.pyplot as plt
from ctg.query import get, list_series

list_series()

In [ ]:
df = get(["FRED:DGS10", "FRED:DGS2"], start="2000-01-01")
df.head()

In [ ]:
df["2s10s"] = df["FRED:DGS10"] - df["FRED:DGS2"]
df[["FRED:DGS10", "FRED:DGS2", "2s10s"]].tail()

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

df[["FRED:DGS10", "FRED:DGS2"]].plot(ax=ax1, lw=1)
ax1.set_title("US Treasury yields")
ax1.set_ylabel("Percent")
ax1.legend(["10Y", "2Y"])
ax1.grid(alpha=0.3)

df["2s10s"].plot(ax=ax2, lw=1, color="#444")
ax2.axhline(0, color="red", lw=0.8, ls="--")
ax2.fill_between(df.index, df["2s10s"], 0, where=df["2s10s"] < 0, color="red", alpha=0.15)
ax2.set_title("2s10s spread (10Y − 2Y)")
ax2.set_ylabel("Percent")
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## How a future Claude Code session uses this

Whoever opens this repo in Claude Code can ask things like:

> *"Plot 5y rolling correlation between SPX and the 10y yield since 2010."*

and the agent just imports `ctg.query.get`, picks the series IDs from `list_series()`, and goes. No fetching code, no API keys at chart time — the warehouse already has it.